# Yayımlanmış v3 ve aynı ölçülerle üç komşu
24 toprakta karşılaştırma. Kaynak: [vitalclick, sabit sürüm](https://github.com/vitalclick/Predicting-Soil-Grain-Size/tree/0808d05def19e03a6478410827a644bfd411ba81). Yerelde work/vitalclick altında src ve cache klasörleri gerekir.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.preprocessing import StandardScaler

root = Path("/kaggle/input") if Path("/kaggle/input").exists() else next(p for p in [Path("work/soil-data"), Path("../work/soil-data")] if p.exists())
data = next(root.rglob("Training_labels_updated.csv")).parent
labels = pd.read_csv(data / "Training_labels_updated.csv").set_index("sample_id")
cameras = pd.read_csv(data / "ppm_updated.csv")
photos = sorted(data.joinpath("Training-All_Photos_updated").rglob("*.jpg"))
print(f"{len(labels)} numune, {len(photos)} eğitim fotoğrafı")

24 numune, 127 eğitim fotoğrafı


## 1. Fotoğrafı bir avuç sayıyla anlatmak
Bütün fotoğrafların merkezinden yaklaşık 100 × 100 mm alan alıyoruz. Kamera tablosundaki piksel/mm değerini gerçek dosyanın çözünürlüğüne uyarlıyoruz. Bu kalibrasyon, çekim uzaklığının kamera tablosunda varsayılana yakın olduğunu kabul eder.

Bu alanı griye çevirip 400 × 400 piksele getiriyoruz. Yatay ve dikey yönde, birbirinden 0,25 mm ile 8 mm arasında uzakta olan noktaların ne kadar farklı aydınlıkta göründüğünü ölçüyoruz. Böylece her fotoğrafta aynı altı fiziksel uzaklığı karşılaştırıyoruz. Her numunenin fotoğraflarını, her ölçünün ortanca değerini alarak birleştiriyoruz.

In [2]:
def physical_patch(path, camera):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        w, h = image.size
        pixels_per_mm = camera.ppm * max(w, h) / max(camera.width, camera.height)
        side = round(100 * pixels_per_mm)
        assert 0 < side <= min(w, h)
        x, y = (w - side) // 2, (h - side) // 2
        return image.crop((x, y, x + side, y + side)).convert("RGB").resize((400, 400), Image.Resampling.LANCZOS)

def texture(patch):
    gray = np.asarray(patch.convert("L"), dtype=float)
    low, high = np.percentile(gray, [5, 95])
    gray = np.clip((gray - low) / max(high - low, 1.0), 0, 1)
    values = []
    for offset in [1, 2, 4, 8, 16, 32]:
        horizontal = np.abs(gray[:, offset:] - gray[:, :-offset]).mean()
        vertical = np.abs(gray[offset:, :] - gray[:-offset, :]).mean()
        values.append((horizontal + vertical) / 2)
    return values

camera_by_name = {c.phone.replace(" ", "_").lower(): c for c in cameras.itertuples()}
rows = []
examples = {}
for path in photos:
    phone, sample, number = path.stem.rsplit("_", 2)
    patch = physical_patch(path, camera_by_name[phone.lower()])
    rows.append([sample, *texture(patch)])
    examples.setdefault(sample, patch)
features = pd.DataFrame(rows, columns=["sample_id", "0.25", "0.5", "1", "2", "4", "8"])
features = features.groupby("sample_id").median()
assert set(features.index) == set(labels.index)
features = features.loc[labels.index]

## v3 doğrulaması
Yazarın iki doğrusal PLS modelini birleştiren v3 kodunu ve paylaştığı hazır ölçüleri kullanıyoruz. Her turda saklanan toprağın tüm çekimleri ve türetilmiş görüntüleri eğitim dışında kalır. Eğitimde türetilmiş çekimler de vardır, sınavda yalnızca özgün çekimler vardır.

In [3]:
import sys
from scipy.stats import spearmanr
repo = next(p for p in [Path("work/vitalclick"), Path("../work/vitalclick")] if p.is_dir())
sys.path.insert(0, str((repo / "src").resolve()))
import run as published
import gsd
cache = pd.read_csv(repo / "cache/features.csv")
cols = [c for c in cache if c not in published.META]
original = cache[(cache.split == "train") & (cache.aug == 0)].copy()
assert sorted(original.file) == sorted(p.name for p in photos)
assert all(r.sample_id == Path(r.file).stem.rsplit("_", 2)[1] for r in original.itertuples())
assert list(labels.columns) == gsd.COLS
published.LINEAR_ONLY = True
v3 = published.loso(cache, labels, pd.DataFrame(gsd.curve_to_logq(labels.values), index=labels.index), cols)
print(v3.mean().round(3))

R1-pls1     39.472
R2-pls1     38.359
ENSEMBLE    38.758
dtype: float64


## Ölçüler değiştiğinde komşular değişiyor mu?
Yazarın v3’te kullandığı iki özellik listesinin birleşimini alıyoruz. İki model için de özgün çekimleri numune başına ortancayla birleştiriyoruz. Saklanan toprağı çıkardıktan sonra her özelliği standartlaştırıp en yakın üç numunenin eğrisini ortalıyoruz. Laboratuvar komşularını yalnızca seçimleri değerlendirmek için kullanıyoruz.

In [4]:
sets = published.feature_sets(cols)
keys = list(dict.fromkeys(sets["R1"] + sets["R2"]))
external = original.groupby("sample_id")[keys].median().loc[labels.index]
assert len(external) == 24 and np.isfinite(external).all().all()
print("Özellik sayıları:", {k: len(v) for k,v in sets.items()}, "Birleşim:", len(keys))
records = []
for sample in labels.index:
    others = labels.index.drop(sample)
    truth = labels.loc[sample].to_numpy()
    lab_distance = gsd.emd(labels.loc[others].values, truth)
    lab_order = others[np.argsort(lab_distance, kind="stable")]
    for name, frame in [("Sade", features), ("Açık ölçüler", external)]:
        train = frame.loc[others]
        scaler = StandardScaler().fit(train)
        dist = np.linalg.norm(scaler.transform(train) - scaler.transform(frame.loc[[sample]]), axis=1)
        idx = np.argsort(dist, kind="stable")[:3]
        chosen = others[idx]
        curve = labels.loc[chosen].mean().to_numpy()
        assert sample not in chosen and len(set(chosen)) == 3
        records.append([sample, name, gsd.emd(truth, curve)[0], len(set(chosen)&set(lab_order[:3])),
                        lab_distance[idx].mean(), spearmanr(dist, lab_distance).statistic, ", ".join(chosen)])
detail = pd.DataFrame(records, columns=["Numune","Model","EMD","İsabet","Komşu EMD","Sıra uyumu","Komşular"])
assert len(detail) == 48
print(detail.groupby("Model").agg({"EMD":"mean","İsabet":"sum","Komşu EMD":"mean","Sıra uyumu":"mean"}).round(3))
per_soil = detail.pivot(index="Numune", columns="Model", values="EMD")
per_soil["v3"] = v3["ENSEMBLE"]
assert round(per_soil.Sade.mean(),3) == 44.744
print(per_soil.round(2).to_string())
for name in ["Açık ölçüler","v3"]:
    delta = per_soil[name] - per_soil.Sade
    print(name, "iyileşen / eşit / kötüleşen:", (delta < -1e-8).sum(), (delta.abs() <= 1e-8).sum(), (delta > 1e-8).sum())
print(detail[detail.Numune.isin(["H038","H405"])].round(2).to_string(index=False))

Özellik sayıları: {'R1': 30, 'R2': 15} Birleşim: 30
                 EMD  İsabet  Komşu EMD  Sıra uyumu
Model                                              
Açık ölçüler  39.743      20     47.144       0.660
Sade          44.744      22     53.043       0.574
Model   Açık ölçüler    Sade      v3
Numune                              
F827           62.26   63.98   31.43
G190           35.31   24.85   17.74
H030           46.10   44.43   18.56
H037           60.27   60.27   91.45
H038           93.28   93.28  119.66
H126           50.64   14.28   28.85
H181           19.27   20.89   20.28
H183           11.00   28.41   19.59
H366            9.83    8.11   21.75
H367           17.29    7.67   20.44
H368           20.47   14.50   13.67
H371           11.07   14.03   23.83
H372            7.27    7.27   35.34
H374           21.28  128.87   52.40
H405           82.64  133.53   74.21
H493           22.60   27.70   56.24
H516           49.26   90.35   44.57
H549           32.21   13.97   37.31


## 21 ortak toprakta telefon değişimi

Özgün fotoğrafların dosya adlarını, toprak ve telefon kimliklerini eşleştiriyoruz. Aynı toprak iki telefonla da çekilmişse sınava giriyor. Her turda bir toprağı saklayıp kalan 20 toprağın tek telefondaki ölçüleriyle üç komşuyu buluyoruz. İkinci telefonun da aynı toprağa ait ölçüleriyle ayrıca tahmin yapıyoruz. İki yönün ortalamasında her toprağın ağırlığı eşit.

In [5]:
scales = features.columns.tolist()
frame = pd.DataFrame(rows, columns=["sample_id", *scales])
frame["file"] = [p.name for p in photos]
matched = original.set_index("file").loc[frame.file]
assert np.array_equal(matched.sample_id, frame.sample_id)
assert matched.phone.tolist() == [p.stem.rsplit("_", 2)[0] for p in photos]
open_columns = keys.copy()
assert len(open_columns) == 30
frame[open_columns] = matched[open_columns].to_numpy()
columns = scales + open_columns
phones = ["Motorola_Edge", "Samsung_A52"]
frame["telefon"] = [p.stem.rsplit("_", 2)[0] for p in photos]
assert frame["sample_id"].tolist() == [p.stem.rsplit("_", 2)[1] for p in photos]
per_phone = frame.groupby(["telefon", "sample_id"])[columns].median()
motorola, samsung = [per_phone.loc[p].index for p in phones]
paired = labels.index[labels.index.isin(motorola) & labels.index.isin(samsung)]
assert len(paired) == 21 and paired.is_unique
widths = np.diff(np.log10(labels.columns.astype(float)))

def emd(a, b):
    return np.abs(a - b)[:-1] @ widths

records = []
for name, keys in [("Sade", scales), ("30 ölçü", open_columns)]:
    for source, other in [phones, phones[::-1]]:
        for sample in paired:
            train = per_phone.loc[source, keys].loc[paired.drop(sample)]
            assert len(train) == 20 and sample not in train.index
            scaler = StandardScaler().fit(train)
            reference = scaler.transform(train)
            curves = []
            for viewed in [source, other]:
                query = per_phone.loc[viewed, keys].loc[[sample]]
                distance = np.linalg.norm(reference - scaler.transform(query), axis=1)
                nearest = train.index[np.argsort(distance, kind="stable")[:3]]
                curves.append(labels.loc[nearest].mean().to_numpy())
            truth = labels.loc[sample].to_numpy()
            same_error, cross_error = [emd(curve, truth) for curve in curves]
            records.append([name, source, sample, same_error, cross_error, cross_error - same_error, emd(*curves)])
results = pd.DataFrame(records, columns=["Model", "Eğitim telefonu", "Numune", "Aynı", "Diğer", "Hata farkı", "Tahminler arası EMD"])
assert len(results) == 84 and not results.duplicated(["Model", "Eğitim telefonu", "Numune"]).any()
metrics = ["Aynı", "Diğer", "Hata farkı", "Tahminler arası EMD"]
summary = results.groupby(["Model","Eğitim telefonu"])[metrics].mean()
assert np.allclose(summary.loc["Sade", ["Aynı", "Diğer"]], [[45.98,49.10],[45.01,45.26]], atol=0.005, rtol=0)
print(summary.round(3).to_string())
print("İki eğitim yönünün eşit ağırlıklı ortalaması")
print(results.groupby("Model")[metrics].mean().round(3).to_string())

paired_errors = results.groupby(["Numune","Model"])["Diğer"].mean().unstack()
gain = paired_errors["Sade"] - paired_errors["30 ölçü"]
assert len(gain) == 21 and np.isfinite(gain).all()
print("İki yön ortalamasında iyileşen/eşit/kötüleşen:", (gain>1e-8).sum(), (gain.abs()<=1e-8).sum(), (gain< -1e-8).sum())
print(paired_errors.assign(Kazanç=gain).round(2).sort_values("Kazanç").to_string())

                           Aynı   Diğer  Hata farkı  Tahminler arası EMD
Model   Eğitim telefonu                                                 
30 ölçü Motorola_Edge    36.089  58.949      22.859               40.847
        Samsung_A52      44.229  40.239      -3.990               26.143
Sade    Motorola_Edge    45.985  49.095       3.111               19.912
        Samsung_A52      45.005  45.265       0.259               27.548
İki eğitim yönünün eşit ağırlıklı ortalaması
           Aynı   Diğer  Hata farkı  Tahminler arası EMD
Model                                                   
30 ölçü  40.159  49.594       9.435               33.495
Sade     45.495  47.180       1.685               23.730
İki yön ortalamasında iyileşen/eşit/kötüleşen: 7 1 13
Model   30 ölçü    Sade  Kazanç
Numune                         
H666      62.72   40.12  -22.59
H493      42.37   20.66  -21.70
H617      41.67   20.24  -21.43
H126      61.58   40.54  -21.04
H405      88.77   67.72  -21.04
F827      7